### Dataset and Task Metadata

In [ ]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="wine_world_cost",
    dataset_year="2023",
    domain_str="business & marketing",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/elvinrustam/wine-dataset",
    download_description="""
There are many different wine datasets available, but this one is a comprehensive collection of wine-related data that includes information on various wine characteristics and text. So we stick to it.

kaggle datasets download elvinrustam/wine-dataset && unzip wine-dataset.zip && rm wine-dataset.zip
mkdir -p local-data-warehouse/wine_world_cost && mv WineDataset.csv local-data-warehouse/wine_world_cost
""",
    # References
    academic_reference_bibtex=r"""@misc{Rustamov2023WineDataset,
  author = {Elvin Rustamov},
  title  = {Wine Dataset},
  year   = {2023},
  howpublished = {\url{https://www.kaggle.com/datasets/elvinrustam/wine-dataset}},
  note   = {Kaggle dataset}
}
""",
    academic_reference_bibtex_key="Rustamov2023WineDataset",
    license="CC0: Public Domain",
    data_tags=["IID"],
    curation_comments="""
We start with the Kaggle version. We resolve several issues from the web scrapper.

- We create a task to predict the price per bottle. We remove the 11 rows that have price information per case or each.
- Create the price target by parsing the price column.
- We map capacity to numeric values in ML.
- We parse the ABV to a numeric value
- We parse the vintage to a numeric date value, ensuring that each wine maps to the oldest year in its special syntax.
- We log scale the target, as the price distribution is very skewed.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Price",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "WineDataset.csv")
print("Loaded data shape:", df.shape)

df = df[df["Per bottle / case / each"] == "per bottle"].reset_index(drop=True)

df["Price"] = df["Price"].apply(lambda x: x.replace("£", "").replace(" per bottle", "")).astype(float)
df["ABV%"] = df["ABV"].apply(lambda x: str(x).replace("ABV ", "").replace("%", "")).astype(float)

df["Capacity"] = df["Capacity"].map({
    "75CL": 750,
    "37.5CL": 375,
    "1.5LTR": 1500,
    "750ML": 750,
    "150CL": 1500,
    "50CL": 500,
    "70CL": 700,
    "500ML": 500,
    "375ML": 375,
    "300CL": 3000,
}).astype(float)
df["VintageYear"] = pd.to_datetime(df["Vintage"].apply(lambda x: x.split("/")[0]).replace("NV", np.nan)).dt.year

df = df.drop(columns=[
    # Constant
    "Per bottle / case / each",
    # Meaning of feature is missing, and it is unclear how to interpret it
    "Unit",
    # Copied columns
    "ABV",
    "Vintage",
])

as_cat_type = [
    "Closure",
    "Type",
    "Style",
]
as_string_type = [
    "Title",
    "Description",
    "Grape",
    "Secondary Grape Varieties",
    "Country",
    "Characteristics",
    "Region",
    "Appellation",
]

for c in as_string_type:
    nan_mask = df[c].isna()
    df.loc[nan_mask, c] = np.nan
    df[c] = df[c].astype("string")

df[as_cat_type] = df[as_cat_type].astype("category")

# Log scale target
df["Price"] = np.log(df["Price"])
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [ ]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    problem_type=task_mold.problem_type,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)

In [ ]:
# Sample Rows
df_head

In [ ]:
# Feature Summary
summary

In [ ]:
# Numeric Feature Statistics
numeric_stats

In [ ]:
# Categorical Feature Statistics
cat_stats

In [ ]:
# Target Distribution
target_df

## Task Curation

In [ ]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

In [ ]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

split_random_state = 4267  # one seed for every split in the benchmark (== curation_recommendations.SPLIT_RANDOM_STATE)
splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
        random_state=split_random_state,
    ),
    split_random_state=split_random_state,
)

## Bundle

In [ ]:
from data_foundry.curation_container import CuratedContainer

curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
)
print(curated_data.describe())

## Bundle Checks


In [ ]:
from data_foundry.bundle_checks import run_bundle_checks

report = run_bundle_checks(
    curated_data,
    # Accepted on purpose (state the reason for each):
    ignore=[],
)
report.raise_if_errors()

## Export

In [ ]:
from data_foundry.bundle_checks import verify_saved_container

save_path = curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

verify_saved_container(save_path, container=curated_data).raise_if_errors()